In [526]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from typing import List
from tqdm import tqdm
import random
import seaborn as sns
from typing import Tuple
import pandas as pd
from rectools import Columns, dataset

In [527]:
df_users = pd.read_csv('KION_DATASET/data_original/users.csv')
df_interactions = pd.read_csv('KION_DATASET/interactions.csv')
df_items = pd.read_csv('KION_DATASET/data_original/items.csv')

<h2> Предобработка данных

In [528]:
df_interactions.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250.0,72.0
1,699317,1659,2021-05-29,8317.0,100.0
2,656683,7107,2021-05-09,10.0,0.0
3,864613,7638,2021-07-05,14483.0,100.0
4,964868,9506,2021-04-30,6725.0,100.0


In [529]:
df_users.head()

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


In [530]:
df_items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [531]:
df_interactions['last_watch_dt'].unique()
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')
df_interactions = df_interactions.dropna(subset=['last_watch_dt'])

In [532]:
from datetime import datetime, timedelta
test_size_days = 10

# Тестовый промежуток времени равен 10 дней
max_date = df_interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=test_size_days)

df_interactions = df_interactions[(df_interactions['watched_pct']>50.0)] # уберем фильмы с просмотром менее 50%
df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]


In [533]:
train_sorted = df_interactions_train.sort_values(['user_id', 'last_watch_dt'])
    
train = train_sorted.groupby('user_id').tail(15) # у пользователей в train берем последние 15 фильмов

In [534]:
def get_active_users_ids(df, min_interactions=2):
    """Возвращает ID пользователей, у которых больше минимального числа взаимодействий."""
    counts = df['user_id'].value_counts()
    return set(counts[counts >= min_interactions].index)

# 1. Находим активных пользователей в каждом наборе данных
active_train_ids = get_active_users_ids(train)
active_test_ids = get_active_users_ids(df_interactions_test)

# 2. Находим пересечение (пользователи, активные и в train, и в test)
common_active_users = active_train_ids & active_test_ids

# 3. Фильтруем исходные данные, оставляя только общих активных пользователей
df_interactions_train = train[train['user_id'].isin(common_active_users)]
df_interactions_test = df_interactions_test[df_interactions_test['user_id'].isin(common_active_users)]

In [535]:
def precision(recommended_list, bought_list):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    return precision


def precision_at_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    bought_list = bought_list 
    recommended_list = recommended_list[:k]
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    
    return precision

In [536]:
def ap_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(recommended_list, bought_list)
    
    if sum(flags) == 0:
        return 0
    
    sum_ = 0
    for i in range(0, k-1):
        if flags[i] == True:
            p_k = precision_at_k(recommended_list, bought_list, k=i+1)
            sum_ += p_k
            
    result = sum_ / sum(flags)
    
    return result

def map_k(recommended_list, bought_list, k=5, u=1):
    
    if u == 1:
        return ap_k(recommended_list[u-1], bought_list[u-1], k=5)
    
    sum = 0
    for i in range(0, u):
        ap_k_map = ap_k(recommended_list[i], bought_list[i], k=5)
        sum += ap_k_map

    result = sum / u
    
    return result

In [537]:
data_test_group = df_interactions_test.groupby(df_interactions_test['user_id'])['item_id'].unique().reset_index()
data_test_group.columns = ['user_id', 'item_id']
data_test_group

,user_id,item_id
0,259,"[10509, 10772]"
1,655,"[11188, 5199]"
2,753,"[9327, 8350]"
3,835,"[5434, 11640, 10878]"
4,960,"[8636, 12770, 6809]"
...,...,...
5049,1096841,"[10219, 3620]"
5050,1097296,"[3006, 16087]"
5051,1097444,"[13650, 12841, 13099, 1124, 12250, 2483]"
5052,1097459,"[11844, 7793]"


In [538]:
userids = data_test_group['user_id'].values
 
userids_test = np.arange(len(userids))

<h2>LightFM

In [539]:
df_interactions_test.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'last_watch_dt': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)


df_interactions_train.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'last_watch_dt': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)

C:\Users\kanze\AppData\Local\Temp\ipykernel_21528\1242639805.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_interactions_train.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item,


In [540]:

dataset_tr = dataset.Dataset.construct(df_interactions_train)

In [541]:
from rectools.models import LightFMWrapperModel
from lightfm import LightFM

model = LightFMWrapperModel(
        # внутри модели указываем параметр no_components
        # это размезность эмбеддингов, которые выучит модель
        model=LightFM(no_components = 10,  random_state=42),
        )


In [542]:
users_unique = df_interactions_test[Columns.User].unique()
model.fit(dataset_tr)
recos = model.recommend(
    users=users_unique,
    dataset=dataset_tr,
    k = 10,
    filter_viewed= True 
)

recos

,user_id,item_id,score,rank
0,73446,9728,3.879552,1
1,73446,13865,3.787980,2
2,73446,3734,3.764881,3
3,73446,8636,3.471192,4
4,73446,142,3.441287,5
...,...,...,...,...
50535,612350,142,3.764843,6
50536,612350,7571,3.759116,7
50537,612350,12173,3.670367,8
50538,612350,4457,3.666616,9


In [543]:
result_recom = recos.groupby(recos['user_id'])['item_id'].unique().reset_index()
result_recom.columns = ['user_id', 'item_id']
result_recom

,user_id,item_id
0,259,"[9728, 13865, 3734, 10440, 8636, 7571, 142, 44..."
1,655,"[9728, 13865, 3734, 10440, 142, 8636, 7571, 12..."
2,753,"[9728, 13865, 3734, 10440, 8636, 142, 7571, 44..."
3,835,"[13865, 3734, 10440, 142, 8636, 7571, 4457, 44..."
4,960,"[13865, 9728, 3734, 8636, 7571, 142, 4457, 378..."
...,...,...
5049,1096841,"[9728, 13865, 3734, 10440, 8636, 7571, 142, 44..."
5050,1097296,"[9728, 13865, 3734, 10440, 8636, 7571, 142, 44..."
5051,1097444,"[9728, 13865, 3734, 10440, 8636, 7571, 142, 44..."
5052,1097459,"[9728, 3734, 10440, 8636, 142, 7571, 7582, 129..."


In [544]:
print('map_k =', map_k(result_recom['item_id'], data_test_group['item_id'], k=10, u=len(data_test_group)))

map_k = 0.05738029283735659
